# Payment Delay Analysis

## Project Overview
Analyze invoice and payment records to understand late payment patterns by customer, invoice size, geography, and payment terms. This is a descriptive analytics project.

## Learning Objectives
- Calculate and profile payment delay metrics across dimensions
- Identify customers and segments with chronic late payments
- Understand how invoice size and payment terms affect delay
- Detect seasonal patterns in payment behavior

## Problem Statement
A B2B company wants to reduce its Days Sales Outstanding (DSO). Which customer segments pay late? Do larger invoices take longer? Are certain payment terms associated with worse compliance?

## Why This Project Matters
Late payments strain cash flow, increase collection costs, and signal customer financial stress. Understanding patterns enables proactive credit management and targeted collection strategies.

## Dataset Overview
Synthetic invoice dataset: ~5K invoices over 2 years with customer segments, regions, payment terms, and actual payment dates.

## Dataset Source and License Notes
- Synthetic data for educational purposes
- No license restrictions

## Environment Setup

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import matplotlib
matplotlib.use('Agg')

## Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style='whitegrid', palette='Set2')
np.random.seed(42)
print('Imports OK')

## Configuration / Constants

In [ ]:
import os
SAVE_DIR = os.path.dirname(os.path.abspath('__file__'))
print(f'Save dir: {SAVE_DIR}')

## Dataset Download or Loading

In [ ]:
np.random.seed(42)
n = 5000
customers = [f'CUST{i:04d}' for i in range(200)]
customer_ids = np.random.choice(customers, n)
segments = {'Enterprise': 0.20, 'Mid-Market': 0.35, 'SMB': 0.45}
cust_segment = {c: np.random.choice(list(segments.keys()), p=list(segments.values())) for c in customers}
regions = np.random.choice(['North', 'South', 'East', 'West', 'Central'], n, p=[0.22, 0.18, 0.20, 0.25, 0.15])
terms = np.random.choice(['Net 15', 'Net 30', 'Net 45', 'Net 60'], n, p=[0.15, 0.40, 0.25, 0.20])
term_days = {'Net 15': 15, 'Net 30': 30, 'Net 45': 45, 'Net 60': 60}

invoice_dates = pd.date_range('2023-01-01', '2024-12-31', freq='D')
inv_dates = np.array([pd.Timestamp(d) for d in np.random.choice(invoice_dates, n)])

amounts = np.clip(np.random.lognormal(8.5, 1.0, n), 500, 200000).round(2)

# Payment delay: baseline from terms + random delay + segment effect
segment_delay = {'Enterprise': -2, 'Mid-Market': 3, 'SMB': 8}
delays = np.array([
    max(0, term_days[t] + segment_delay[cust_segment[c]] + int(np.random.normal(5, 12)))
    for t, c in zip(terms, customer_ids)
])
due_dates = inv_dates + pd.to_timedelta([term_days[t] for t in terms], unit='D')
pay_dates = inv_dates + pd.to_timedelta(delays, unit='D')
days_late = np.maximum(0, ((pay_dates - due_dates) / pd.Timedelta(days=1)).astype(int))

df = pd.DataFrame({
    'InvoiceID': [f'INV{i:06d}' for i in range(n)],
    'CustomerID': customer_ids,
    'Segment': [cust_segment[c] for c in customer_ids],
    'Region': regions,
    'InvoiceDate': inv_dates,
    'DueDate': due_dates,
    'PaymentDate': pay_dates,
    'Amount': amounts,
    'PaymentTerms': terms,
    'DaysToPayment': delays,
    'DaysLate': days_late,
})
df['OnTime'] = (df['DaysLate'] == 0).astype(int)
df['YearMonth'] = df['InvoiceDate'].dt.to_period('M')

print(f'Dataset shape: {df.shape}')
print(f'On-time rate: {df["OnTime"].mean():.1%}')
print(f'Avg days late (when late): {df.loc[df["DaysLate"]>0, "DaysLate"].mean():.1f}')
df.head()

## Data Validation Checks

In [ ]:
print(f'Missing values: {df.isnull().sum().sum()}')
print(f'Invoice date range: {df["InvoiceDate"].min().date()} to {df["InvoiceDate"].max().date()}')
print(f'Amount range: ${df["Amount"].min():,.2f} - ${df["Amount"].max():,.2f}')
print(f'\nSegment distribution:\n{df["Segment"].value_counts()}')
print(f'\nPayment terms distribution:\n{df["PaymentTerms"].value_counts()}')

## Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

df['DaysLate'].hist(bins=50, ax=axes[0,0], edgecolor='black', alpha=0.7, color='coral')
axes[0,0].set_title('Days Late Distribution (All Invoices)')
axes[0,0].axvline(df['DaysLate'].median(), color='red', linestyle='--', label=f'Median: {df["DaysLate"].median():.0f}')
axes[0,0].legend()

df.groupby('Segment')['DaysLate'].mean().sort_values().plot.barh(ax=axes[0,1], edgecolor='black', color='steelblue')
axes[0,1].set_title('Avg Days Late by Segment')

df.groupby('PaymentTerms')['OnTime'].mean().sort_values().plot.bar(ax=axes[1,0], edgecolor='black', color='green')
axes[1,0].set_title('On-Time Rate by Payment Terms')
axes[1,0].tick_params(axis='x', rotation=0)

monthly_late = df.groupby('YearMonth')['DaysLate'].mean()
monthly_late.plot(ax=axes[1,1], marker='o', color='purple')
axes[1,1].set_title('Monthly Avg Days Late Trend')
axes[1,1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'eda_overview.png'), dpi=100, bbox_inches='tight')
plt.show()

## Payment Delay by Invoice Size

In [ ]:
df['AmountBucket'] = pd.qcut(df['Amount'], q=5, labels=['Very Small', 'Small', 'Medium', 'Large', 'Very Large'])
size_stats = df.groupby('AmountBucket').agg(
    count=('InvoiceID', 'count'),
    avg_days_late=('DaysLate', 'mean'),
    on_time_rate=('OnTime', 'mean'),
    avg_amount=('Amount', 'mean'),
).round(3)
print('Payment Delay by Invoice Size:')
print(size_stats)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
size_stats['avg_days_late'].plot.bar(ax=axes[0], edgecolor='black', color='steelblue')
axes[0].set_title('Avg Days Late by Invoice Size')
axes[0].tick_params(axis='x', rotation=45)

size_stats['on_time_rate'].plot.bar(ax=axes[1], edgecolor='black', color='green')
axes[1].set_title('On-Time Rate by Invoice Size')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'size_analysis.png'), dpi=100, bbox_inches='tight')
plt.show()

## Chronic Late Payers

In [ ]:
cust_stats = df.groupby('CustomerID').agg(
    invoices=('InvoiceID', 'count'),
    avg_days_late=('DaysLate', 'mean'),
    on_time_rate=('OnTime', 'mean'),
    total_amount=('Amount', 'sum'),
    segment=('Segment', 'first'),
).round(3)
chronic = cust_stats[(cust_stats['invoices'] >= 5) & (cust_stats['on_time_rate'] < 0.3)].sort_values('avg_days_late', ascending=False)
print(f'Chronic late payers (5+ invoices, <30% on-time): {len(chronic)}')
print(chronic.head(10))

fig, ax = plt.subplots(figsize=(10, 5))
ax.scatter(cust_stats['invoices'], cust_stats['avg_days_late'], alpha=0.4, c=cust_stats['on_time_rate'], cmap='RdYlGn', edgecolor='gray', s=30)
ax.set_xlabel('Number of Invoices')
ax.set_ylabel('Avg Days Late')
ax.set_title('Customer Payment Behavior (color = on-time rate)')
plt.colorbar(ax.collections[0], label='On-Time Rate')
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'chronic_payers.png'), dpi=100, bbox_inches='tight')
plt.show()

## Regional Analysis

In [ ]:
region_stats = df.groupby('Region').agg(
    invoices=('InvoiceID', 'count'),
    avg_days_late=('DaysLate', 'mean'),
    on_time_rate=('OnTime', 'mean'),
    total_outstanding=('Amount', lambda x: x[df.loc[x.index, 'DaysLate'] > 0].sum()),
).round(3).sort_values('avg_days_late', ascending=False)
print('Regional Payment Summary:')
print(region_stats)

fig, ax = plt.subplots(figsize=(10, 5))
region_stats[['avg_days_late']].plot.bar(ax=ax, edgecolor='black', color='coral')
ax.set_title('Avg Days Late by Region')
ax.tick_params(axis='x', rotation=0)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'regional.png'), dpi=100, bbox_inches='tight')
plt.show()

## Segment × Payment Terms Analysis

In [ ]:
cross = df.groupby(['Segment', 'PaymentTerms']).agg(
    avg_late=('DaysLate', 'mean'),
    on_time=('OnTime', 'mean'),
).round(3)
print('Segment × Terms:')
print(cross)

pivot = df.groupby(['Segment', 'PaymentTerms'])['DaysLate'].mean().unstack()
fig, ax = plt.subplots(figsize=(10, 5))
pivot.plot.bar(ax=ax, edgecolor='black')
ax.set_title('Avg Days Late: Segment × Payment Terms')
ax.legend(title='Terms', bbox_to_anchor=(1.05, 1))
ax.tick_params(axis='x', rotation=0)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'segment_terms.png'), dpi=100, bbox_inches='tight')
plt.show()

## Interpretation and Business Insight
- **SMB customers** pay significantly later than Enterprise or Mid-Market, suggesting tighter credit terms or proactive outreach is needed
- **Longer payment terms** (Net 60) don't necessarily improve on-time rates — they just shift the baseline
- **Invoice size** has moderate impact on delay — very large invoices tend to take slightly longer
- **Chronic late payers** are identifiable and concentrated — targeted collections could improve DSO significantly
- **Regional differences** exist but are modest compared to segment effects

## Limitations
- Synthetic data with simplified delay logic
- No partial payment data — real invoices may have multiple payment installments
- No dispute or credit memo information
- Customer segment assignment is static (real segmentation evolves)
- No cash flow impact modeling

## How to Improve This Project
- Use real AP/AR data for authentic payment behavior analysis
- Add aging bucket analysis (30/60/90/120+ days)
- Include dispute and credit memo data
- Build a payment prediction model (will this invoice be late?)
- Add cash flow impact analysis and DSO forecasting

## Production Considerations
- Automate DSO tracking dashboards with alerts for chronic late payers
- Integrate with ERP for real-time aging analysis
- Build predictive payment scoring for new invoices
- Implement dynamic credit policies based on payment history

## Common Mistakes
- Averaging days late without separating on-time from late (skews results)
- Not accounting for payment terms when comparing customers
- Ignoring seasonal effects (year-end slow payments)
- Treating all late payments equally regardless of amount at risk

## Mini Challenge / Exercises
1. Calculate DSO (Days Sales Outstanding) for each quarter. Is DSO improving or worsening?
2. Which payment terms have the highest dollar-weighted on-time rate (weighted by invoice amount)?
3. If you could convert the top 10 chronic late payers to on-time, how much would average DSO improve?
4. Is there a day-of-week effect on payment timing?

## Final Summary / Key Takeaways
- Payment delay analysis is essential for cash flow management and credit risk
- Customer segmentation reveals systematic differences in payment behavior
- Chronic late payers are concentrated — targeted action has outsized impact
- Payment terms, invoice size, and region all influence delay patterns
- Proactive monitoring beats reactive collections for DSO improvement